<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/engr330_charleston_southern_university/overvoltage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overvoltage Hosting Capacity Example

This notebook illustrates a simple hosting capacity workflow using OpenDSS and Python.

In [34]:
!pip install py-dss-interface
!pip install py-dss-toolkit
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 42 (delta 3), reused 42 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 186.51 KiB | 7.77 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/opendss-python-examples/feeder_models/8bus/opendss-python-examples/feeder_models/8bus/opendss-python-examples/feeder_models/8bus/opendss-python-examples


In [35]:
import pathlib
import numpy as np
import py_dss_interface
from py_dss_toolkit import dss_tools

# --- Helper functions ---
STEP_KW = 100
MAX_KW = 10000
OV_THRESHOLD = 1.05

def add_gen(dss, gen_bus, gen_kv):
    for gen in gen_bus:
        dss.text(f"new generator.{gen} phases=3 bus1={gen_bus[gen]} kv={gen_kv[gen]} "
                 f"kw=0.0001 pf=1 Vminpu=0.7 Vmaxpu=1.2")

def increase_gen(dss, gen_kw):
    for gen, kw in gen_kw.items():
        dss.text(f"Edit generator.{gen} kw={kw}")

def solve_powerflow(dss):
    dss.text("solve")

def check_overvoltage_violation(dss):
    voltages = dss.circuit.buses_vmag_pu
    return max(voltages) > OV_THRESHOLD

def set_load_level_condition(dss, load_mult):
    dss.text(f"set loadmult={load_mult}")

def result_centralized_info(bus, metric, hc_value, load_condition, existing_gen, device_type):
    return f"""Summary of the Centralized Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = {load_condition}
    Existing Gen considered = {existing_gen}

Hosting Capacity:
    Bus = {bus}
    Value = {hc_value} kW
    Device Type = {device_type}
    Metric = {metric}"""

## Define feeder and create OpenDSS object and connected to dss_tools

In [36]:
start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "8bus" / "Master.dss"
dss = py_dss_interface.DSS()

dss_tools.update_dss(dss)

print(f"Using feeder file: {dss_file}")

Using feeder file: /content/opendss-python-examples/feeder_models/8bus/opendss-python-examples/feeder_models/8bus/opendss-python-examples/feeder_models/8bus/opendss-python-examples/feeder_models/8bus/Master.dss


## Compile feeder and set operating condition

In [37]:
dss.text(f'compile "{dss_file}"')

load_mult = 0.2
set_load_level_condition(dss, load_mult=load_mult)

print(f"Load multiplier set to {load_mult}.")

Load multiplier set to 0.2.


## Select the bus where generation will be connected

In [38]:
bus = "4"

dss.circuit.set_active_bus(bus)
kv = dss.bus.kv_base * np.sqrt(3)

gen_bus = {"gen": dss.bus.name}
gen_kv = {"gen": kv}

print(f"Selected bus: {dss.bus.name}")
print(f"Generator base voltage: {kv:.4f} kV")

Selected bus: 4
Generator base voltage: 12.4700 kV


## Add the generator

In [39]:
add_gen(dss, gen_bus, gen_kv)

print("Generator added successfully.")

Generator added successfully.


## Increase generation until overvoltage is detected

In [40]:
hosting_capacity_value = None

i = 0
while i * STEP_KW < MAX_KW:
    i += 1
    i_kw = i * STEP_KW
    gen_kw = {"gen": i_kw}

    increase_gen(dss, gen_kw)
    solve_powerflow(dss)

    if check_overvoltage_violation(dss):
        hosting_capacity_value = (i - 1) * STEP_KW
        print(f"Overvoltage violation detected at {i_kw} kW.")
        break

if hosting_capacity_value is None:
    hosting_capacity_value = MAX_KW
    print("No overvoltage violation detected within the tested range.")

Overvoltage violation detected at 2200 kW.


## Display the result

In [41]:
result = result_centralized_info(
    bus,
    "Overvoltage",
    hosting_capacity_value,
    "offpeak",
    False,
    "Generator"
)

print(result)

Summary of the Centralized Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = offpeak
    Existing Gen considered = False

Hosting Capacity:
    Bus = 4
    Value = 2100 kW
    Device Type = Generator
    Metric = Overvoltage


## Checking the feeder with the generator sized to the hosting capacity value

In [42]:
i_kw = hosting_capacity_value
gen_kw = {"gen": i_kw}

increase_gen(dss, gen_kw)
solve_powerflow(dss)

In [43]:
dss_tools.interactive_view.voltage_profile()

In [44]:
sub_marker = dss_tools.interactive_view.circuit_get_bus_marker("0", "square", 12, "black")
gen_marker = dss_tools.interactive_view.circuit_get_bus_marker("4", "circle", 10, "red")
dss_tools.interactive_view.circuit_plot(parameter="active power", bus_markers=[sub_marker, gen_marker])

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)